# Branch II: the period-2 orbit born at the end of the satellite's period-doubling bubble (23/10/2026)

Follow-up of `satellite_start_bubble_edge.ipynb` (section 4). At $\varepsilon_d = 10\,g_2$ the bubble's chaotic attractor dies in a boundary crisis at $g_c \simeq 101.4$ MHz, and the escaping trajectories land on a period-2 orbit
($\langle|a(nT_a)|\rangle = 3.0$ or $3.9$ up to about 106 MHz). Two candidates were left open:
1. **branch II**, the period-2 orbit born at the reverse period doubling of the satellite (end of the bubble, 103.9 MHz), never computed;
2. **the $\omega_a/2$ orbit** of `Hill_method_cat_fold.ipynb` (period $2T_a$, dominated by the odd harmonics, stable on about 101.2–106.8 MHz at $\varepsilon_d = 10.12\,g_2$).

Here, at $\varepsilon_d = 10\,g_2$ (the row of `bubble_crisis_lifetime.jl`):
1. both edges of the bubble are located by bisection on the satellite;
2. **branch II** (and branch I, from the lower edge) is followed by harmonic balance over $2T_a$, starting from the bifurcation point along the odd null vector of the period-2 Jacobian, by pseudo-arclength in $g_2$;
3. the $\omega_a/2$ orbit is computed (time integration, then harmonic balance) and continued in $g_2$ through its folds;
4. both are compared with the escape destinations of `bubble_crisis_lifetime_22_10_2026.json`;
5. branch II is compared with the basin of the satellite beyond the bubble (section 3 of `satellite_start_bubble_edge`).

No Lyapunov map: harmonic balance, Floquet multipliers, and one time integration of $4000\,T_a$. The whole notebook runs headless in about 2 minutes.

**Conventions.** As in `satellite_start_bubble_edge.ipynb`: $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$, $\kappa_b = \pi/10.4$ (code convention), $\tau = \omega_a t$, $T_a = 2\pi$ in $\tau$.
Period-2 harmonic balance: basis over $\tau_d = \tau/2$ ($M\to M/4$, $C\to C/2$, drive on harmonic 4); odd harmonics of $\tau_d$ are the $\omega_a/2$ components, zero on a period-1 orbit.
Multipliers over $2T_a$ for period-2 orbits. Section: $a(\tau = 2\pi k) = (x_a + i\,dx_a/d\tau)/(\sqrt2\varphi_a)$, as in `bubble_crisis_lifetime.jl`. The partner orbit ($a\to-a$, shift by $T_a/2$) is sampled at $\tau = \pi + 2\pi k$.

In [ ]:
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import root
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *
warnings.filterwarnings("ignore", category=RuntimeWarning)

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/"

w_a = 25.338776456203686
w_b = 2*w_a
phi_a, phi_b = 0.11, 0.204
E_J = 37.12*2*np.pi*1.5
kappa_b = np.pi/10.4                 # code convention
G0 = E_J*phi_a**2*phi_b              # g2 = s G0, s = sin(eps_p)
R = 10.0                             # eps_d / g2 (row of bubble_crisis_lifetime.jl)
n = 2
M = np.diag([1.0, 0.25])
K = np.eye(2)
N_H, N_H2 = 16, 32                   # harmonics: period 1, period 2 (N_H2 = 2 N_H, so the satellite embeds exactly)
G_C = 101.4                          # crisis of the bubble's attractor (satellite_start_bubble_edge, section 4)

def MHz(g):
    return g/2/np.pi*1e3
g2_max = MHz(G0)

## Harmonic balance, Floquet multipliers, continuation
`HB`, `HB2`, `orbit`, `monodromy`, `satellite_hb`, ... are copied from `satellite_start_bubble_edge.ipynb` (same as `beyond_cusp_regimes.ipynb`). Added here:
- `newton`: plain Newton corrector (`hybr` stalls when the guess is already converged, as during a bisection);
- `embed`: a period-1 solution written in the period-2 basis (harmonic $k$ → $2k$);
- `arclength`: pseudo-arclength in $(z, p)$ with $p = g_2/(100\text{ MHz})$, tangent from the SVD of the extended Jacobian;
- `fourier_seed`: a time-integrated period-2 state turned into harmonic-balance coefficients;
- `section`: $a$ on the section $\tau = 0 \bmod 2\pi$ and for the partner orbit.

In [ ]:
class HB:
    def __init__(self, N_H, kfac=1.0, N=None):
        self.N_H, self.N = N_H, N or 64*N_H
        self.C = np.diag([0.0, kfac*kappa_b/w_b])
        self.L = self.build_L(M, self.C)
        t = np.linspace(0, 2*np.pi, self.N, endpoint=False)
        k = np.arange(1, N_H + 1)
        Phi = np.empty((self.N, 2*N_H + 1))
        Phi[:, 0] = 1/np.sqrt(2)
        Phi[:, 1::2] = np.sin(np.outer(t, k))
        Phi[:, 2::2] = np.cos(np.outer(t, k))
        self.Gamma = np.kron(Phi, np.eye(n))
        self.Gamma_pinv = (2/self.N)*self.Gamma.T
        self.drive_harmonic = 2

    def build_L(self, M_, C_):
        L = np.zeros((n*(2*self.N_H + 1),)*2)
        L[:n, :n] = K
        for k in range(1, self.N_H + 1):
            A, Bk = K - k**2*M_, k*C_
            s_, c_ = 2*k - 1, 2*k
            L[s_*n:(s_+1)*n, s_*n:(s_+1)*n] = A
            L[s_*n:(s_+1)*n, c_*n:(c_+1)*n] = -Bk
            L[c_*n:(c_+1)*n, s_*n:(s_+1)*n] = Bk
            L[c_*n:(c_+1)*n, c_*n:(c_+1)*n] = A
        return L

    def set_s(self, s, r=R):
        self.B = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])          # (beta_a', beta_b')
        self.b_ext = np.zeros(n*(2*self.N_H + 1))
        self.b_ext[2*self.drive_harmonic*n + 1] = -np.sqrt(2)*r*s*G0*phi_b/w_b

    def X(self, z):
        return (self.Gamma @ z).reshape(self.N, n).T

    def residual(self, z):
        X = self.X(z)
        Fnl = -np.sqrt(2)*self.B[:, None]*(np.cos(np.sqrt(2)*(X[0] + X[1])) - 1)[None, :]
        return self.L @ z + self.Gamma_pinv @ Fnl.T.reshape(-1) - self.b_ext

    def jacobian(self, z):
        X = self.X(z)
        sn = np.sin(np.sqrt(2)*(X[0] + X[1]))
        JG = np.einsum('j,ab,jbm->jam', 2*sn, np.outer(self.B, [1, 1]),
                       self.Gamma.reshape(self.N, n, -1)).reshape(n*self.N, -1)
        return self.L + self.Gamma_pinv @ JG

    def solve(self, z0, tol=1e-12):
        sol = root(self.residual, z0, jac=self.jacobian, method="hybr", tol=tol, options={"maxfev": 100*(len(z0) + 1)})
        if not sol.success:
            raise RuntimeError(sol.message)
        return sol.x


class HB2(HB):
    # period-2 orbit: basis over tau_d = tau/2
    def __init__(self, N_H, kfac=1.0, N=None):
        super().__init__(N_H, kfac, N)
        self.L = self.build_L(M/4, self.C/2)
        self.drive_harmonic = 4


def orbit(z, N_H, t):
    # x and dx/dt of both modes (t in the basis time of the HB object)
    k = np.arange(1, N_H + 1)
    out = []
    for zz in (z[::2], z[1::2]):
        s_, c_ = zz[1::2], zz[2::2]
        x = zz[0]/np.sqrt(2) + np.sin(np.outer(t, k)) @ s_ + np.cos(np.outer(t, k)) @ c_
        v = np.cos(np.outer(t, k)) @ (k*s_) - np.sin(np.outer(t, k)) @ (k*c_)
        out.append((x, v))
    return out

def strobe(z, N_H, tau=0.0):
    # period-1 orbit: state [x_a, x_b, v_a, v_b] at time tau
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, np.array([tau]))
    return np.array([x_a[0], x_b[0], v_a[0], v_b[0]])

def rotating_frame(z, N_H, tau, period=1):
    # a_R = i a e^{i tau}, b_R = b e^{2 i tau} (limit_cycles.ipynb); tau in units of omega_a t
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, tau/period)
    v_a, v_b = v_a/period, v_b/period                                   # d/dtau
    return 1j*(x_a + 1j*v_a)/(np.sqrt(2)*phi_a)*np.exp(1j*tau), (x_b + 1j*v_b/2)/(np.sqrt(2)*phi_b)*np.exp(2j*tau)

def odd_content(z, N_H):
    # odd harmonics of the memory in the period-2 basis: zero on a period-1 orbit
    za = z[::2]
    return np.sqrt(sum(za[2*k - 1]**2 + za[2*k]**2 for k in range(1, N_H + 1, 2)))

def monodromy(hb, z, period=1):
    Minv, base = np.linalg.inv(M), np.outer(hb.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hb.N_H, np.atleast_1d(t)/period)
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ hb.C                       # hb.C is in tau units for both bases
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi*period), np.eye(4).reshape(-1), method="DOP853", rtol=1e-10, atol=1e-12)
    return np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))

def instability(mu):
    # type of the leading multiplier
    m = mu[np.argmax(np.abs(mu))]
    if abs(m) < 1:
        return "stable"
    if abs(m.imag) > 1e-6:
        return "NS"
    return "PD" if m.real < 0 else "saddle"

def full_rhs(s, kfac=1.0, r=R):
    beta = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])
    eps_d_p, kp = r*s*G0*phi_b/w_b, kfac*kappa_b/w_b
    def f(t, Y):
        x_a, x_b, v_a, v_b = Y
        g = np.cos(np.sqrt(2)*(x_a + x_b)) - 1
        return [v_a, v_b, -x_a + np.sqrt(2)*beta[0]*g, 4*(-x_b - np.sqrt(2)*eps_d_p*np.cos(2*t) + np.sqrt(2)*beta[1]*g - kp*v_b)]
    return f


# analytic satellite (App. "Satellites", quadratic order), only used as the first seed
yb_grid = np.linspace(1e-3, 3.0, 3000)
def satellite_hb(s, r=R):
    kp = kappa_b/w_b
    ba, bb = 2*E_J*s*phi_a**2/w_a, 2*E_J*s*phi_b**2/w_b
    ed, bS = r*s*G0*phi_b/w_b, ba + bb
    gam = 1 + bb/(ba*(1 + kp**2))
    eeff = lambda y: np.sqrt(np.maximum(ed**2 - 4*kp**2*y**2, 0.0))
    def y_of(yb):
        y = yb + ba*ed/(2*bb)
        for _ in range(30):
            y = yb + ba*eeff(y)/(2*bb)
        return y
    def u_of(yb):
        eta = 2*bb*yb
        return (np.sqrt((1 + eta)**2 + 4*eta**2) - (1 + eta))/(2*eta)
    def x2_of(yb):
        u = u_of(yb)
        return (2*(1 + u)*yb**2 - eeff(y_of(yb))/bb)/(1 + 2*u)
    def resid(yb):
        u = u_of(yb)
        return yb*(1 + u) - 2*bS*((1 + (1 + u)**2/2)*yb**2 + (1 + u**2)*x2_of(yb))
    rv = resid(yb_grid)
    for k in np.where(rv[:-1]*rv[1:] <= 0)[0]:
        lo, hi = yb_grid[k], yb_grid[k + 1]
        for _ in range(60):
            mid = (lo + hi)/2
            if resid(lo)*resid(mid) <= 0:
                hi = mid
            else:
                lo = mid
        yb = (lo + hi)/2
        x2 = x2_of(yb)
        if x2 <= 0:
            continue
        u, y, x = u_of(yb), y_of(yb), np.sqrt(x2)
        if 2*kp*abs(y) > ed:
            continue
        th = -np.arcsin(2*kp*y/ed)/2
        a1 = -(1 + u)*yb/(2*gam)
        return dict(a0=x*np.exp(1j*th), a1=a1, am1=-ba*eeff(y)/(2*bb)*np.exp(2j*th),
                    b0=y*np.exp(2j*th), bm1=u*x*np.exp(3j*th), b2=bb/ba*a1)

def seed_z(S, N_H):
    za, zb = np.zeros(2*N_H + 1), np.zeros(2*N_H + 1)
    za[0] = 2*np.real(S["a1"])
    za[1], za[2] = np.sqrt(2)*np.imag(S["a0"]), np.sqrt(2)*np.real(S["a0"])
    za[3], za[4] = np.sqrt(2)*np.imag(S["am1"]), np.sqrt(2)*np.real(S["am1"])
    zb[0] = 2*np.real(S["b2"])
    zb[3], zb[4] = np.sqrt(2)*np.imag(S["b0"]), np.sqrt(2)*np.real(S["b0"])
    zb[5], zb[6] = np.sqrt(2)*np.imag(S["bm1"]), np.sqrt(2)*np.real(S["bm1"])
    z = np.zeros(n*(2*N_H + 1)); z[::2], z[1::2] = za, zb
    return z

In [ ]:
def newton(hb, z, tol=1e-11, maxit=30):
    for _ in range(maxit):
        r_ = hb.residual(z)
        if np.linalg.norm(r_) < tol:
            return z
        z = z - np.linalg.solve(hb.jacobian(z), r_)
    if np.linalg.norm(hb.residual(z)) < 1e3*tol:
        return z
    raise RuntimeError("Newton did not converge")

def embed(z1, N1, N2):
    # period-1 coefficients in the period-2 basis: harmonic j of tau -> harmonic 2j of tau_d
    z2 = np.zeros(n*(2*N2 + 1)); z2[:n] = z1[:n]
    for j in range(1, min(N1, N2//2) + 1):
        z2[(4*j - 1)*n:4*j*n] = z1[(2*j - 1)*n:2*j*n]
        z2[4*j*n:(4*j + 1)*n] = z1[2*j*n:(2*j + 1)*n]
    return z2

ODD = np.zeros(n*(2*N_H2 + 1), bool)                    # coefficients of the odd harmonics (both modes)
for k in range(1, N_H2 + 1, 2):
    ODD[(2*k - 1)*n:(2*k + 1)*n] = True

PS = 100.0                                              # continuation parameter p = g2 / (100 MHz)
def extended(hb2):
    def setp(p): hb2.set_s(p*PS/g2_max)
    def F(y):
        setp(y[-1]); return hb2.residual(y[:-1])
    def dF(y, h=1e-7):
        setp(y[-1]); Jz, r_ = hb2.jacobian(y[:-1]), hb2.residual(y[:-1])
        setp(y[-1] + h)
        return np.column_stack([Jz, (hb2.residual(y[:-1]) - r_)/h])
    return F, dF

def arclength(hb2, y, t, ds0=2e-3, ds_max=3e-3, g_lo=60.0, g_hi=140.0, n_max=5000, stop_odd=True):
    # pseudo-arclength from y = (z, p) along t; stops when g2 leaves [g_lo, g_hi] or (stop_odd) the odd harmonics vanish again
    F, dF = extended(hb2)
    def corrector(y0, t, ds):
        w = y0 + ds*t
        for _ in range(20):
            G = np.append(F(w), t @ (w - y0) - ds)
            if np.linalg.norm(G) < 1e-11:
                return w
            w = w - np.linalg.solve(np.vstack([dF(w), t]), G)
        if np.linalg.norm(np.append(F(w), t @ (w - y0) - ds)) < 1e-9:
            return w
        raise RuntimeError
    ds, hist, end = ds0, [y.copy()], "n_max reached"
    for it in range(n_max):
        try:
            yn = corrector(y, t, ds)
        except (RuntimeError, np.linalg.LinAlgError):
            ds /= 2
            if ds < 1e-8:
                end = f"step too small at {y[-1]*PS:.2f} MHz"; break
            continue
        tn = np.linalg.svd(dF(yn))[2][-1]
        t = -tn if tn @ t < 0 else tn
        y = yn; hist.append(y.copy()); ds = min(1.3*ds, ds_max)
        g = y[-1]*PS
        if not g_lo < g < g_hi:
            end = f"leaves [{g_lo:g}, {g_hi:g}] MHz"; break
        if stop_odd and it > 20 and odd_content(y[:-1], hb2.N_H) < 1e-5:
            end = f"odd harmonics vanish at {g:.3f} MHz"; break
    return np.array(hist), end

def fourier_seed(y_state, s, N_H2, N=2048):
    # one period 2 T_a from a state on the section, projected on the period-2 basis (tau_d = tau/2)
    tt = np.linspace(0, 4*np.pi, N, endpoint=False)
    sol = solve_ivp(full_rhs(s), (0, 4*np.pi), y_state, t_eval=tt, method="DOP853", rtol=1e-11, atol=1e-12)
    td, k = tt/2, np.arange(1, N_H2 + 1)
    z = np.zeros(n*(2*N_H2 + 1))
    for m in range(2):
        zz = np.empty(2*N_H2 + 1)
        zz[0] = np.sqrt(2)*sol.y[m].mean()
        zz[1::2] = (2/N)*np.sin(np.outer(td, k)).T @ sol.y[m]
        zz[2::2] = (2/N)*np.cos(np.outer(td, k)).T @ sol.y[m]
        z[m::2] = zz
    return z

def state(z, N, tau, period):
    # [x_a, x_b, dx_a/dtau, dx_b/dtau] at tau (x^lambda units of the harmonic balance)
    (x_a, v_a), (x_b, v_b) = orbit(z, N, np.atleast_1d(tau/period))
    return np.array([x_a[0], x_b[0], v_a[0]/period, v_b[0]/period])

def section(z, N, period=2):
    # a on the section tau = 0 mod 2 pi (amp of bubble_crisis_lifetime.jl), and for the partner orbit (tau = pi mod 2 pi)
    taus = 2*np.pi*np.arange(period)
    amp = lambda Y: (Y[0] + 1j*Y[2])/(np.sqrt(2)*phi_a)
    return np.array([amp(state(z, N, t, period)) for t in taus]), np.array([amp(state(z, N, t + np.pi, period)) for t in taus])

def describe(H, N, period=2):
    # multipliers, type, odd content, section, <|a_R|^2> at every point of a branch H (rows (z, p))
    hb_ = HB2(N) if period == 2 else HB(N)
    tt = np.linspace(0, 2*np.pi*period, 300*period, endpoint=False)
    out = dict(g2=H[:, -1]*PS, mu=[], kind=[], odd=[], sec=[], sec_partner=[], a2=[])
    for y in H:
        hb_.set_s(y[-1]*PS/g2_max)
        mu = monodromy(hb_, y[:-1], period); mu = mu[np.argsort(-np.abs(mu))]
        s0, s1 = section(y[:-1], N, period)
        out["mu"].append(mu); out["kind"].append(instability(mu)); out["odd"].append(odd_content(y[:-1], N) if period == 2 else 0.0)
        out["sec"].append(s0); out["sec_partner"].append(s1)
        out["a2"].append(np.mean(np.abs(rotating_frame(y[:-1], N, tt, period)[0])**2))
    return {k: np.array(v) for k, v in out.items()}

def unstable_set(mu):
    # multipliers outside the unit circle: "+" real > 1, "-" real < -1, "c" complex pair (counted once)
    out = [("c" if abs(m.imag) > 1e-6 else ("+" if m.real > 0 else "-")) for m in mu if abs(m) > 1 and m.imag >= -1e-6]
    return "".join(sorted(out)) or "stable"

def transitions(D):
    # changes of the set of unstable multipliers along a branch; g2 of the crossing interpolated on the multiplier that crosses |mu| = 1
    res, sets = [], [unstable_set(m) for m in D["mu"]]
    for i in range(1, len(sets)):
        if sets[i] != sets[i - 1]:
            a, b = np.sort(np.abs(D["mu"][i - 1])), np.sort(np.abs(D["mu"][i]))
            k = np.argmax(np.abs((a > 1).astype(int) - (b > 1).astype(int))) if np.any((a > 1) != (b > 1)) else None
            g = D["g2"][i - 1] + (1 - a[k])*(D["g2"][i] - D["g2"][i - 1])/(b[k] - a[k]) if k is not None else D["g2"][i]
            res.append((g, sets[i - 1], sets[i]))
    return res

# 1. Edges of the bubble at $\varepsilon_d = 10\,g_2$
The satellite (analytic seed at 60 MHz, then natural continuation) is followed up to the end of the bubble. The most negative real multiplier crosses $-1$ at both edges; each crossing is bisected to $10^{-8}$ MHz.

In [ ]:
hb = HB(N_H)
hb.set_s(60/g2_max)
z = hb.solve(seed_z(satellite_hb(60/g2_max), N_H))
SAT = {}
for g in np.arange(60.5, 125.01, 0.5):
    hb.set_s(g/g2_max); z = newton(hb, z); SAT[round(g, 2)] = z.copy()

def pd_margin(g, z):
    # most negative real multiplier + 1 on the satellite at g
    hb.set_s(g/g2_max); z = newton(hb, z)
    mu = monodromy(hb, z); re = mu[np.abs(mu.imag) < 1e-9].real
    return (re.min() if len(re) else 0.0) + 1, z

def bisect_pd(lo, hi):
    z = SAT[round(2*lo)/2]
    v_lo, _ = pd_margin(lo, z)
    for _ in range(35):
        mid = (lo + hi)/2
        v, z = pd_margin(mid, z)
        if (v > 0) == (v_lo > 0):
            lo, v_lo = mid, v
        else:
            hi = mid
    return (lo + hi)/2, pd_margin((lo + hi)/2, z)[1]

g_pd1, z_pd1 = bisect_pd(75.0, 76.0)          # lower edge (period doubling)
g_pd2, z_pd2 = bisect_pd(103.5, 104.0)        # upper edge (reverse period doubling, end of the bubble)
EDGES = json.load(open(OUT_DIR + "satellite_pd_bubble_edges_22_10_2026.json"))
j = EDGES["eps_d_over_g2"].index(R)
print(f"bubble at eps_d = {R:g} g2: {g_pd1:.4f} - {g_pd2:.4f} MHz  (satellite_start_bubble_edge, 0.25 MHz grid: {EDGES['g2_lower_MHz'][j]:.2f} - {EDGES['g2_upper_MHz'][j]:.2f})")
for g in (103.0, 103.5, 103.8, 104.0, 104.2, 105.0):
    hb.set_s(g/g2_max); zz = newton(hb, SAT[round(2*g)/2] if round(2*g)/2 in SAT else z_pd2)
    mu = monodromy(hb, zz)
    print(f"  satellite at {g:6.1f} MHz: multipliers " + "  ".join(f"{m:.3f}" for m in mu[np.argsort(-np.abs(mu))]))

# 2. Branches I and II by harmonic balance over $2T_a$
At a period-1 orbit the period-2 Jacobian is block-diagonal in even and odd harmonics (the nonlinear term is $\pi$-periodic in $\tau_d$). At a period doubling the odd block has a one-dimensional kernel,
the Floquet mode with $\mu = -1$. The new branch leaves the bifurcation point along that null vector at fixed $g_2$; pseudo-arclength then follows it in whichever direction of $g_2$ it goes.
The partner direction ($-v$) gives the same orbit shifted by $T_a$.

In [ ]:
hb2 = HB2(N_H2)
BR = {}
for name, g0, z1 in (("I", g_pd1, z_pd1), ("II", g_pd2, z_pd2)):
    hb2.set_s(g0/g2_max)
    z2e = newton(hb2, embed(z1, N_H, N_H2))
    J = hb2.jacobian(z2e)
    S, Vt = np.linalg.svd(J[np.ix_(ODD, ODD)])[1:]
    v = np.zeros(len(z2e)); v[ODD] = Vt[-1]
    print(f"branch {name}: even-odd coupling of the Jacobian {max(np.abs(J[np.ix_(ODD, ~ODD)]).max(), np.abs(J[np.ix_(~ODD, ODD)]).max()):.1e}; "
          f"two smallest singular values of the odd block {S[-2]:.2e}, {S[-1]:.2e}")
    t0 = time.time()
    H, end = arclength(hb2, np.append(z2e, g0/PS), np.append(v, 0.0), g_hi=140.0)
    BR[name] = describe(H, N_H2); BR[name]["H"] = H
    g = BR[name]["g2"]
    print(f"  {len(H)} points, {g.min():.2f} - {g.max():.2f} MHz, g2 monotonic along the branch: {np.all(np.diff(g) > 0) or np.all(np.diff(g) < 0)}; {end} ({time.time() - t0:.0f} s)")
    print("  unstable multipliers (+: mu > 1, -: mu < -1, c: complex pair): " + "; ".join(f"{a} -> {b} at {gg:.2f} MHz" for gg, a, b in transitions(BR[name])))

d2 = BR["II"]["g2"] - g_pd2
for i in (2, 5, 10, 30, 80):
    print(f"branch II, g2 - g_PD = {d2[i]:7.3f} MHz: odd harmonics {BR['II']['odd'][i]:.4f}, /sqrt(g2 - g_PD) = {BR['II']['odd'][i]/np.sqrt(d2[i]):.4f};  "
          f"multipliers " + "  ".join(f"{m:.3f}" for m in BR["II"]["mu"][i]))

# 3. The $\omega_a/2$ orbit at $\varepsilon_d = 10\,g_2$
Same recipe as `Hill_method_cat_fold.ipynb` (where it was found at $\varepsilon_d = 10.12\,g_2$): from the satellite at 103.5 MHz (just inside the bubble) plus a $10^{-4}$ kick, $4000\,T_a$ of time integration, then one period $2T_a$ projected on the period-2 basis and corrected by Newton.
The branch is followed by pseudo-arclength in both directions of $g_2$, through its folds.

In [ ]:
G_SEED = 103.5
hb.set_s(G_SEED/g2_max); z_sat = newton(hb, SAT[G_SEED])
# integrate in chunks of 1000 T_a until the section has settled on period 2 (the chaotic transient at 103.5 MHz lasts a few hundred 1/kappa_b)
t0, y_ = time.time(), strobe(z_sat, N_H) + 1e-4
for chunk in range(60):
    sol = solve_ivp(full_rhs(G_SEED/g2_max), (0, 2*np.pi*1000), y_, method="DOP853", rtol=1e-10, atol=1e-12, t_eval=2*np.pi*np.arange(990, 1001))
    y_ = sol.y[:, -1]
    a_tail = (sol.y[0] + 1j*sol.y[2])/(np.sqrt(2)*phi_a)
    if abs(a_tail[-1] - a_tail[-3]) < 1e-8 and abs(a_tail[-1] - a_tail[-2]) > 0.1:
        break
print(f"time integration ({time.time() - t0:.0f} s): settled on period 2 after {1000*(chunk + 1)} T_a; |a(nT_a)| over the last iterates {np.round(np.abs(a_tail[-4:]), 4)}; "
      f"|a_N - a_(N-2)| = {abs(a_tail[-1] - a_tail[-3]):.1e}")
hb2.set_s(G_SEED/g2_max)
z_half = fourier_seed(sol.y[:, -1], G_SEED/g2_max, N_H2)
print(f"residual of the Fourier seed {np.linalg.norm(hb2.residual(z_half)):.1e}")
z_half = newton(hb2, z_half)
assert odd_content(z_half, N_H2) > 0.3, "the seed did not converge to the omega_a/2 orbit"
print(f"odd harmonics {odd_content(z_half, N_H2):.3f} (branch II at the same g2 does not exist; branch I: {np.interp(G_SEED, BR['I']['g2'], BR['I']['odd']):.3f})")

F_, dF_ = extended(hb2)
y0 = np.append(z_half, G_SEED/PS)
t_ = np.linalg.svd(dF_(y0))[2][-1]
parts = []
for sgn in (-1, 1):
    tt = t_ if np.sign(t_[-1]) == sgn else -t_
    H, end = arclength(hb2, y0, tt, stop_odd=False, n_max=3000)
    parts.append(H); print(f"direction {'down' if sgn < 0 else 'up'} in g2: {len(H)} points, {end}")
H_half = np.vstack([parts[0][::-1], parts[1][1:]])
OM = describe(H_half, N_H2); OM["H"] = H_half
g = OM["g2"]
i_turn = np.where(np.diff(np.sign(np.diff(g))) != 0)[0] + 1
print("folds at g2 = " + ", ".join(f"{g[i]:.3f}" for i in i_turn) + " MHz")
i_sn = i_turn[0]                                         # saddle-node where the orbit is born
NODE = {k: v[i_sn:] for k, v in OM.items()}            # from the fold, through the stable part
SADDLE = {k: v[:i_sn + 1][::-1] for k, v in OM.items()} # from the fold, the other way
g_sn = g[i_sn]
print("node branch: " + "; ".join(f"{a} -> {b} at {gg:.2f} MHz" for gg, a, b in transitions(NODE)))
print(f"saddle branch (from the fold at {g_sn:.3f} MHz up to {SADDLE['g2'].max():.1f} MHz): types {sorted(set(SADDLE['kind']))}, "
      f"max|mu| = {np.abs(SADDLE['mu'][np.argmin(np.abs(SADDLE['g2'] - G_C)), 0]):.1f} at {G_C} MHz")
print("saddle branch: " + "; ".join(f"{a} -> {b} at {gg:.2f} MHz" for gg, a, b in transitions(SADDLE)))

# 4. Comparison with the escape destinations of the crisis
`bubble_crisis_lifetime_22_10_2026.json` stores, for every escaping trajectory, the period of the final orbit and $\langle|a|\rangle$ over its points on the section $\tau = 0 \bmod 2\pi$.
A period-2 orbit and its partner ($a\to-a$, shifted by $T_a/2$) are both attractors and have different $\langle|a|\rangle$ on that section, so each period-2 attractor shows up as **two** values.

In [ ]:
C = json.load(open(OUT_DIR + "bubble_crisis_lifetime_22_10_2026.json"))
def on_branch(D, g, key):
    i = np.argmin(np.abs(D["g2"] - g))
    return np.mean(np.abs(D[key][i])), D["kind"][i], abs(D["g2"][i] - g)

print(" g2 (MHz) | escape destinations (period: <|a|>, number of runs)            | omega_a/2 node: <|a|>, partner, type | branch II: <|a|>, partner")
DEST = []
for g in sorted({x["g2_MHz"] for x in C["runs"]}):
    rr = [x for x in C["runs"] if x["g2_MHz"] == g and x["t_esc_Ta"] is not None]
    if not rr:
        continue
    cnt = {}
    for x in rr:
        key = (x["period"], round(x["abs_a_end"], 2)); cnt[key] = cnt.get(key, 0) + 1
    DEST += [(g, p, a, c) for (p, a), c in cnt.items()]
    s0, kind, dg = on_branch(NODE, g, "sec"); s1 = on_branch(NODE, g, "sec_partner")[0]
    b2 = f"{on_branch(BR['II'], g, 'sec')[0]:.2f}, {on_branch(BR['II'], g, 'sec_partner')[0]:.2f}" if g > g_pd2 else "(does not exist)"
    txt = ", ".join(f"{p}: {a:.2f} ({c})" for (p, a), c in sorted(cnt.items(), key=lambda kv: -kv[1])[:4])
    print(f"  {g:6.1f} | {txt:62s} | {s0:.2f}, {s1:.2f}, {kind:6s} | {b2}")
DEST = np.array(DEST, float)

# 5. Branch II and the basin of the satellite beyond the bubble
Section 3 of `satellite_start_bubble_edge.ipynb` kicked the exact satellite towards the analytic seed (all grid points above the end of the bubble, $\varepsilon_d/g_2 = 4.5$–13.5):
the kick comes back for 83 % of the points at $\epsilon = 0.003$, 64 % at 0.01 and 0 % at 0.03 ($x^\lambda$ units). A subcritical period doubling leaves the satellite surrounded by the period-2 saddle,
whose stable manifold is the edge of the basin. Below: the distance on the section between branch II (both of its points) and the satellite, in the same units.

In [ ]:
BEY = json.load(open(OUT_DIR + "satellite_start_beyond_bubble_22_10_2026.json"))
eps_kick, back = np.array(BEY["eps_kick"]), np.array([p["kick_returns"] for p in BEY["points"]]).mean(axis=0)
H2 = BR["II"]["H"]
D_SAT = []
z = z_pd2
for y in H2[1::3]:
    g = y[-1]*PS
    hb.set_s(g/g2_max); z = newton(hb, z)
    ys = strobe(z, N_H)
    D_SAT.append((g, *(np.linalg.norm(state(y[:-1], N_H2, t, 2) - ys) for t in (0.0, 2*np.pi))))
D_SAT = np.array(D_SAT)
for g in (104, 106, 110, 115, 120):
    i = np.argmin(np.abs(D_SAT[:, 0] - g))
    print(f"g2 = {D_SAT[i, 0]:6.2f} MHz: distance branch II - satellite on the section {D_SAT[i, 1]:.4f} / {D_SAT[i, 2]:.4f}")
print("kicks of section 3: " + ", ".join(f"eps = {e:g}: {100*b:.0f} % back" for e, b in zip(eps_kick, back)))

# 6. Figures
Left: bifurcation diagram at $\varepsilon_d = 10\,g_2$, $\langle|a_R|^2\rangle$ (rotating frame, averaged over the period) against $g_2$, line style by type of the leading multiplier. Middle: same, odd harmonics of the memory (zero on the satellite).
Right: the section $\langle|a(2\pi k)|\rangle$ of the period-2 orbits and of their partners, with the escape destinations of the crisis runs (crosses, size $\propto$ number of runs).
Bottom: orbits at 105 MHz in the rotating frame, and the distance between branch II and the satellite against the kicks of section 3.

In [ ]:
KIND_STYLE = {"stable": dict(ls="-", lw=1.2), "saddle": dict(ls="--", lw=0.8), "PD": dict(ls=":", lw=1.0), "NS": dict(ls="-.", lw=0.8)}
COL = {"satellite": Perso_palette_float[5], "I": Perso_palette_float[2], "II": "crimson", "node": Perso_palette_float[3], "saddle": "darkorange"}
LAB = {"satellite": "satellite (period 1)", "I": "branch I (from the period doubling)", "II": "branch II (from the reverse period doubling)",
       "node": r"$\omega_a/2$ orbit", "saddle": r"$\omega_a/2$ orbit, saddle side of the fold"}

gs_sat = np.array(sorted(SAT)); gs_sat = gs_sat[(gs_sat >= 60) & (gs_sat <= 125)]
SATD = describe(np.array([np.append(SAT[g], g/PS) for g in gs_sat]), N_H, period=1)
CURVES = {"satellite": SATD, "I": BR["I"], "II": BR["II"], "node": NODE, "saddle": SADDLE}

def plot_kinds(ax, x, y, kind, color):
    for k_, st in KIND_STYLE.items():
        m = kind == k_
        if m.any():
            ax.plot(x, np.where(m, y, np.nan), color=color, **st)

fig, axes = plt.subplots(2, 3, figsize=(3.3*figsize_car[0], 2.3*figsize_car[1]))
for name, D in CURVES.items():
    plot_kinds(axes[0, 0], D["g2"], D["a2"], D["kind"], COL[name])
    plot_kinds(axes[0, 1], D["g2"], D["odd"], D["kind"], COL[name])
    if name != "satellite":
        plot_kinds(axes[0, 2], D["g2"], np.abs(D["sec"]).mean(axis=1), D["kind"], COL[name])
        plot_kinds(axes[0, 2], D["g2"], np.abs(D["sec_partner"]).mean(axis=1), D["kind"], COL[name])
m2 = DEST[:, 1] == 2
axes[0, 2].scatter(DEST[m2, 0], DEST[m2, 2], s=2*DEST[m2, 3], marker="x", color="k", lw=0.6, zorder=5, label="escape destinations, period 2")
axes[0, 2].scatter(DEST[~m2, 0], DEST[~m2, 2], s=2*DEST[~m2, 3], marker="+", color="0.4", lw=0.6, zorder=5, label="escape destinations, other periods")
for ax in axes[0]:
    ax.axvspan(g_pd1, g_pd2, color=Perso_palette_float[3], alpha=0.08, lw=0)
    ax.axvline(G_C, color="k", lw=0.6, ls=":")
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_xlim(70, 125)
axes[0, 0].set_ylabel(r"$\langle|a_R|^2\rangle$"); axes[0, 1].set_ylabel("odd harmonics of $x_a$ (period-2 basis)")
axes[0, 2].set_ylabel(r"$\langle|a(2\pi k)|\rangle$ (orbit and partner)"); axes[0, 2].set_xlim(98, 122); axes[0, 2].legend(fontsize=4.5, loc="upper left")
axes[0, 0].set_title(rf"$\varepsilon_d = {R:g}\,g_2$; shaded: bubble; dotted: crisis $g_c$", fontsize=6)
handles = [mpl.lines.Line2D([], [], color=COL[k], lw=1.2) for k in CURVES] + [mpl.lines.Line2D([], [], color="k", **st) for st in KIND_STYLE.values()]
labels = [LAB[k] for k in CURVES] + ["stable", r"saddle ($\mu > 1$)", r"period doubling ($\mu < -1$)", "Neimark–Sacker"]
axes[0, 1].legend(handles, labels, fontsize=4.5, loc="upper left")

G_SHOW = 105.0
tt1, tt2 = np.linspace(0, 2*np.pi, 300), np.linspace(0, 4*np.pi, 600)
for name, D in CURVES.items():
    i = np.argmin(np.abs(D["g2"] - G_SHOW))
    if abs(D["g2"][i] - G_SHOW) > 0.5:
        continue
    zz = SAT[G_SHOW] if name == "satellite" else D["H"][i, :-1]
    a_R, b_R = rotating_frame(zz, N_H, tt1) if name == "satellite" else rotating_frame(zz, N_H2, tt2, period=2)
    ls = "-" if D["kind"][i] == "stable" else "--"
    for sign, alpha in ((1, 1.0), (-1, 0.3)):
        axes[1, 0].plot(sign*a_R.real, sign*a_R.imag, ls, color=COL[name], lw=0.8, alpha=alpha)
    axes[1, 1].plot(b_R.real, b_R.imag, ls, color=COL[name], lw=0.8)
for ax, lab in ((axes[1, 0], "a"), (axes[1, 1], "b")):
    ax.set_aspect("equal"); ax.set_xlabel(rf"Re ${lab}_R$"); ax.set_ylabel(rf"Im ${lab}_R$")
axes[1, 0].set_title(rf"orbits at {G_SHOW:g} MHz (dashed: unstable; light: partner $a\to-a$)", fontsize=6)
ax = axes[1, 2]
ax.plot(D_SAT[:, 0], D_SAT[:, 1], color=COL["II"], lw=1, label="branch II - satellite, on the section")
ax.plot(D_SAT[:, 0], D_SAT[:, 2], color=COL["II"], lw=0.6, ls="--")
for e, b in zip(eps_kick, back):
    ax.axhline(e, color="0.5", lw=0.6, ls=":")
    ax.text(121.5, e*1.08, f"kick {e:g}: {100*b:.0f} % back", fontsize=4.5, ha="right")
ax.set_yscale("log"); ax.set_xlim(g_pd2, 122); ax.set_ylim(1e-3, 0.15)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"distance ($x^\lambda$ units)"); ax.legend(fontsize=4.5, loc="lower right")
ax.set_title("satellite basin beyond the bubble", fontsize=6)
plt.tight_layout()
plt.savefig(FIG_DIR + "period2_branch_II_eps_d10_23_10_2026.pdf", bbox_inches="tight")
plt.show()

## Export
`period2_branches_eps_d10_23_10_2026.json`: edges of the bubble; for each period-2 family (branch I, branch II, $\omega_a/2$ node and saddle sides) $g_2$, type, the four multipliers, odd content, $\langle|a_R|^2\rangle$, section points of the orbit and of its partner;
the harmonic-balance coefficients every 10 points (`z`, period-2 basis, $N_H = 32$) as seeds for later work.

In [ ]:
def export(D, every=10):
    return {"g2_MHz": D["g2"].tolist(), "kind": D["kind"].tolist(),
            "mu_re": D["mu"].real.tolist(), "mu_im": D["mu"].imag.tolist(), "odd": D["odd"].tolist(), "a2_rot": D["a2"].tolist(),
            "section_re": D["sec"].real.tolist(), "section_im": D["sec"].imag.tolist(),
            "partner_re": D["sec_partner"].real.tolist(), "partner_im": D["sec_partner"].imag.tolist(),
            "z_every": every, "z": D["H"][::every, :-1].tolist()}
json.dump({"description": f"Period-2 orbits at eps_d = {R:g} g2, kappa_b = pi/10.4 (code), harmonic balance over 2 T_a (N_H = {N_H2}, basis tau_d = tau/2). "
                          "Section: a(tau = 2 pi k) = (x_a + i dx_a/dtau)/(sqrt2 phi_a); partner at tau = pi + 2 pi k. From Hill_period2_branch_II.ipynb",
           "eps_d_over_g2": R, "g2_pd_lower_MHz": g_pd1, "g2_pd_upper_MHz": g_pd2, "g2_omega_half_fold_MHz": float(g_sn),
           "branch_I": export(BR["I"]), "branch_II": export(BR["II"]), "omega_half_node": export(NODE), "omega_half_saddle": export(SADDLE)},
          open(OUT_DIR + "period2_branches_eps_d10_23_10_2026.json", "w"))

## Conclusions
**1. Edges of the bubble at $\varepsilon_d = 10\,g_2$:** period doubling at 75.340 MHz, reverse period doubling at 103.883 MHz (0.25 MHz grid of `satellite_start_bubble_edge`: 75.36–103.87).

**2. Branch II is born subcritically.** It exists only **above** the reverse period doubling ($g_2 > 103.88$ MHz), where the satellite is stable, and it is a saddle from birth ($\mu = 1.03$ at 0.09 MHz from the bifurcation).
- Odd harmonics $\simeq 0.0075\sqrt{g_2 - g_\mathrm{PD}}$ near the bifurcation; they stay below 0.026 up to 140 MHz. Branch II is a slightly doubled satellite, not the $\omega_a/2$ orbit (0.55).
- A second multiplier leaves through $-1$ at 114.0 MHz. There is no fold and no reconnection up to 140 MHz ($g_2$ monotonic along the branch).
- Branch I is born **supercritically** at 75.34 MHz, is stable up to its Neimark–Sacker point at 76.80 MHz, then stays unstable (complex pair, odd harmonics $\simeq 0.08$) up to 140 MHz.
  So the two edges of the bubble are of opposite types, and the two period-2 branches of the bubble never meet. The note of 22/10 ("branch II exists for $g_2 < 103.9$") was wrong.

**3. Branch II is the edge of the satellite's basin beyond the bubble.** Its distance to the satellite on the section grows from 0.001 at 103.9 MHz to 0.023 at 120 MHz.
That is the basin width measured by the kicks of section 3 of `satellite_start_bubble_edge` (83 % back at 0.003, 64 % at 0.01, 0 % at 0.03). It explains why a start "slightly off" the satellite (analytic seed, 0.07–0.12 away) never reaches it.
Conversely, going down through 103.88 MHz the satellite loses stability with no nearby stable period-2 orbit (hard loss of stability), so the trajectory leaves for distant attractors.

**4. The escape destination is the $\omega_a/2$ orbit, not branch II.** At $\varepsilon_d = 10\,g_2$ it is born in a saddle-node at 99.970 MHz. The node side is:
- stable on 99.97–100.24 MHz, Neimark–Sacker on 100.24–101.20 MHz;
- stable again on 101.20–106.71 MHz, then Neimark–Sacker;
- period-doubled from 111.5 MHz, with folds at 119.39 and 118.05 MHz.
The saddle side is strongly unstable ($\mu \approx 44$ at 101.4 MHz).
From 101.5 to 106.0 MHz the node reproduces **every** period-2 escape destination of the crisis runs to 0.01 in $\langle|a|\rangle$. The two values (3.0 and 3.9) are the orbit and its partner $a\to-a$, reached in roughly equal numbers.
Above 106.7 MHz the node is unstable, and the escapes go to period-5, period-7/8 and period-1 orbits instead (108–116 MHz).

**5. Still open: what the attractor collides with at $g_c \simeq 101.4$ MHz.** Branch II does not exist there, so it is excluded. The remaining candidates:
- the saddle side of the $\omega_a/2$ fold (exists from 99.97 MHz);
- the edge of the node's basin created when the node restabilises at **101.20 MHz**, 0.2 MHz below $g_c$. If that Neimark–Sacker is subcritical, an unstable torus bounds the node's basin and grows from 101.20 MHz, and a collision with it would put the crisis just above 101.20.

Tests for the next step:
- the type of the Neimark–Sacker at 101.20 MHz (short time integrations near the node at 101.1–101.3 MHz);
- distances in the full 4D section between the attractor and the candidate saddles (the projection on $a$ cannot decide: the attractor cloud fills $|a| \lesssim 8$ with spacing 0.02);
- lifetime runs on 99.97–100.24 MHz, where the node is also stable.